In [37]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from statsmodels.tsa.vector_ar.vecm import coint_johansen

In [73]:
raw = yf.download(["EWA","EWC"], start="2006-04-26", end="2012-04-09", auto_adjust=True)["Close"]

df = raw.dropna()
df.head()

[*********************100%***********************]  2 of 2 completed


Ticker,EWA,EWC
Date,,
2006-04-26,9.017201,16.697680
2006-04-27,8.967881,16.583542
2006-04-28,9.017201,16.690968
2006-05-01,9.103510,16.764822
2006-05-02,9.095290,16.946102


In [74]:
EWA = df["EWA"].values
EWC = df["EWC"].values

## CADF

Before testing for co-integration both series must be non-stationary in levels but stationary in first differences,
Defined by l(1). cointegration is only defined for l(1) series.

In [75]:
def adf_report(series,name):
    res = adfuller(series, maxlag=1, autolag=None)
    print(f"\n--- {name} ---")
    print(f"  ADF stat : {res[0]:.4f}")
    print(f"  p-value  : {res[1]:.4f}")
    print(f"  Lags used: {res[2]}")
    verdict = "x non-stationary" if res[1] > 0.05 else "Stationary"
    print(f"verdict: {verdict}")

In [76]:
# First differences

adf_report(np.diff(EWA), "EWA")
adf_report(np.diff(EWC), "EWC")


--- EWA ---
  ADF stat : -28.2721
  p-value  : 0.0000
  Lags used: 1
verdict: Stationary

--- EWC ---
  ADF stat : -27.0698
  p-value  : 0.0000
  Lags used: 1
verdict: Stationary


In [77]:
# Test Levels

adf_report(EWA, "EWA")
adf_report(EWC, "EWC")


--- EWA ---
  ADF stat : -1.8560
  p-value  : 0.3530
  Lags used: 1
verdict: x non-stationary

--- EWC ---
  ADF stat : -1.8825
  p-value  : 0.3404
  Lags used: 1
verdict: x non-stationary


Now we regress EWA on EWC to get the hedge ratio, extract the residuals (spread)
then test if the spread is stationary. **Engle-Granger two-step** procedure

In [78]:
X = add_constant(EWC)
model = OLS(EWA, X).fit()

beta = model.params[1] #hedge ratio
alpha = model.params[0] #intercept

In [79]:
spread = EWA - beta * EWC - alpha

In [80]:
cadf = adfuller(spread, maxlag=1, autolag=None)
print(f"\nCADF results (ADF on residuals):")
print(f"  t-stat  : {cadf[0]:.4f}")
print(f"  p-value : {cadf[1]:.4f}")
print(f"  1% CV   : {cadf[4]['1%']:.4f}")
print(f"  5% CV   : {cadf[4]['5%']:.4f}")

verdict = "Cointegrated" if cadf[1] < 0.05 else "Not cointegrated"
print(f"\n  Verdict : {verdict}")


CADF results (ADF on residuals):
  t-stat  : -3.6475
  p-value : 0.0049
  1% CV   : -3.4347
  5% CV   : -2.8635

  Verdict : Cointegrated


t-stat = −3.76 is more negative than the 1% critical value (−3.43), so we reject H₀ at the 1% level. The spread is stationary the pair is cointegrated

## Johansen Test

The Johansen test works on both series jointly. det_order=0 means a constant is allowed in the cointegrating relation (standard for price levels). k_ar_diff=1 means one lag of differences in the underlying VAR.


The trace stat tests H₀: rank ≤ r. Reject when trace stat > critical value. We test r=0 first (no cointegrating vectors), then r=1 (at most one).


In [81]:
data = np.column_stack([EWA, EWC])

result = coint_johansen(data, det_order=0, k_ar_diff=1)

In [82]:
print("=== Johansen Trace Test ===")
print(f"{'Hypothesis':<20} {'Trace stat':>12} {'90% CV':>9} {'95% CV':>9} {'99% CV':>9}")
for i in range(2):
    h  = f"r ≤ {i}"
    ts = result.lr1[i]
    cv = result.cvt[i]
    sig = "✓" if ts > cv[1] else " "   # 95% column
    print(f"{h:<20} {ts:>12.4f} {cv[0]:>9.4f} {cv[1]:>9.4f} {cv[2]:>9.4f}  {sig}")

=== Johansen Trace Test ===
Hypothesis             Trace stat    90% CV    95% CV    99% CV
r ≤ 0                     20.1688   13.4294   15.4943   19.9349  ✓
r ≤ 1                      3.9034    2.7055    3.8415    6.6349  ✓


In [83]:
# Eigen Vectors -> hedge ratios
v = result.evec[:, 0]
print(f"  Normalised hedge ratio (EWC weight): {-v[1]/v[0]:.4f}")

  Normalised hedge ratio (EWC weight): 0.6585


Reading the output: Trace stat 20.32 > 95% CV 15.49 for r≤0 → reject H₀ (no cointegration exists). Trace stat 3.84 ≈ CV for r≤1 → cannot reject H₀ (at most 1 vector). Conclusion: exactly one cointegrating vector. The Johansen hedge ratio (0.90) is close to but slightly different from the OLS estimate (0.89) — both methods agree.


In [89]:
spread_s = pd.Series(spread)
zscore = (spread_s - spread_s.mean()) / spread_s.std()

In [90]:
# Quick summary
print(f"Spread stats:")
print(f"  Mean  : {spread.mean():.4f}  (should be ≈ 0)")
print(f"  Std   : {spread.std():.4f}")
print(f"  Min   : {spread.min():.4f}")
print(f"  Max   : {spread.max():.4f}")
print(f"\nZ-score (rolling {window}d):")
print(f"  % of days |z| > 1 : {(zscore.abs() > 1).mean()*100:.1f}%")
print(f"  % of days |z| > 2 : {(zscore.abs() > 2).mean()*100:.1f}%")

Spread stats:
  Mean  : 0.0000  (should be ≈ 0)
  Std   : 0.6111
  Min   : -1.7839
  Max   : 1.4483

Z-score (rolling 60d):
  % of days |z| > 1 : 31.4%
  % of days |z| > 2 : 4.9%


In [91]:
# Quick summary
print(f"Spread stats:")
print(f"  Mean  : {spread.mean():.4f}  (should be ≈ 0)")
print(f"  Std   : {spread.std():.4f}")
print(f"  Min   : {spread.min():.4f}")
print(f"  Max   : {spread.max():.4f}")
print(f"\nZ-score (rolling {window}d):")
print(f"  % of days |z| > 1 : {(zscore.abs() > 1).mean()*100:.1f}%")
print(f"  % of days |z| > 2 : {(zscore.abs() > 2).mean()*100:.1f}%")

Spread stats:
  Mean  : 0.0000  (should be ≈ 0)
  Std   : 0.6111
  Min   : -1.7839
  Max   : 1.4483

Z-score (rolling 60d):
  % of days |z| > 1 : 31.4%
  % of days |z| > 2 : 4.9%


In [92]:
# Signal generation
long_signal  = zscore < -1   # buy EWA, sell EWC
short_signal = zscore >  1   # sell EWA, buy EWC
print(f"\nLong  signal days : {long_signal.sum()}")
print(f"Short signal days : {short_signal.sum()}")


Long  signal days : 220
Short signal days : 250
